Cell 1: Mount Storage

In [0]:
# CELL 1: MOUNT BLOB STORAGE
try:
    dbutils.fs.unmount("/mnt/playbehavior")
except:
    pass

dbutils.fs.mount(
    source="wasbs://<CONTAINER>@<STORAGE_ACCOUNT>.blob.core.windows.net",
    mount_point="/mnt/playbehavior",
    extra_configs={"fs.azure.account.key.<STORAGE_ACCOUNT>.blob.core.windows.net": 
                   "<REDACTED_AZURE_STORAGE_ACCOUNT_KEY>"}
)
print("Mounted /mnt/playbehavior")

/mnt/playbehavior has been unmounted.
Mounted /mnt/playbehavior


Cell 2: Install Packages

In [0]:
%pip install "numpy<2.0" "opencv-python-headless" "motmetrics" "loguru" "accelerate" "timm"
%pip install "transformers" --upgrade
print("Done")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Done


In [0]:
dbutils.library.restartPython()

Cell 4: Imports + GPU

In [0]:
import os, os.path as osp, json, time, gc, glob, shutil
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Literal
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from torchvision import transforms
import torchvision.models as models
from PIL import Image
import timm
from timm.data import resolve_data_config

from transformers import Sam3TrackerVideoModel, Sam3TrackerVideoProcessor
from accelerate import Accelerator

accelerator = Accelerator()
device = accelerator.device
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DEFAULT_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32


Device: cuda:0
GPU: NVIDIA A10-24Q
Memory: 23.7 GB


Cell 5: Load SAM3

In [0]:
from huggingface_hub import login
login(token="<REDACTED_HF_TOKEN>")

processor = Sam3TrackerVideoProcessor.from_pretrained("facebook/sam3")
model = Sam3TrackerVideoModel.from_pretrained("facebook/sam3", torch_dtype=DEFAULT_DTYPE).to(device)
model.eval()
print(f"SAM3: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/845 [00:00<?, ?it/s]

SAM3: 465.8M params


Cell 6: Helpers + Config

In [0]:
# ── Helper functions ──────────────────────────────────────────────────────

def load_bounding_box_prompts(txt_path):
    if not txt_path.startswith("/dbfs"):
        txt_path = osp.join("/dbfs", txt_path.lstrip("/"))
    prompts = {}
    with open(txt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line: continue
            name, coords = line.split(':', 1)
            x, y, w, h = map(float, coords.strip().split(','))
            prompts[name.strip()] = [int(x), int(y), int(x+w), int(y+h)]
    print(f"✓ Loaded {len(prompts)} object prompts from {txt_path}")
    for name, bbox in prompts.items():
        print(f"  {name}: {bbox}")
    return prompts

def prepare_sam3_bbox_format(bboxes):
    return [bboxes]

def get_prompt_path_from_video_path(video_path):
    parts = video_path.rstrip('/').split('/')
    vid_id, vid_name = parts[-2], parts[-1]
    return f"/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Initial Prompts/{vid_id}/{vid_name}/BoundingBoxes_{vid_id}.txt"

def prepare_frames_or_path(video_path):
    if osp.exists(video_path): return video_path
    dbfs_path = osp.join("/dbfs", video_path.lstrip("/"))
    if osp.exists(dbfs_path): return dbfs_path
    raise ValueError(f"Path does not exist: {video_path}")

# ── Paths ─────────────────────────────────────────────────────────────────

# CELL 7: CONFIGURATION

BENCHMARK_VIDEOS = [
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_05_000002/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_11_000028/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_11_000036/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_22_000010/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_28_000113/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_02_000005/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_02_000208/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_10_000060/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_10_000078/video1",
]

BASE_DIR = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3"
FRAMES_DIR = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames"
FPN_DIR = osp.join(BASE_DIR, "FPN_Features")
CHECKPOINT_DIR = osp.join(BASE_DIR, "Checkpoints_FPN")
METRICS_DIR = osp.join(BASE_DIR, "Metrics")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

TRAIN_VIDEO_IDS = [osp.basename(osp.dirname(v)) for v in BENCHMARK_VIDEOS[0:4]]
VAL_VIDEO_IDS   = [osp.basename(osp.dirname(v)) for v in BENCHMARK_VIDEOS[4:5]]
TEST_VIDEO_IDS  = [osp.basename(osp.dirname(v)) for v in BENCHMARK_VIDEOS[5:9]]

print(f"Train: {TRAIN_VIDEO_IDS}")
print(f"Val:   {VAL_VIDEO_IDS}")
print(f"Test:  {TEST_VIDEO_IDS}")

Train: ['2019_11_05_000002', '2019_11_11_000028', '2019_11_11_000036', '2019_11_22_000010']
Val:   ['2019_11_28_000113']
Test:  ['2019_12_02_000005', '2019_12_02_000208', '2019_12_10_000060', '2019_12_10_000078']


In [0]:
# CELL 6.5: EXTRACT BACKBONE FEATURES
# Hook the backbone (before neck) to get [1, 1024, 72, 72] per frame

BACKBONE_DIR = osp.join(BASE_DIR, "Backbone_Features")
os.makedirs(BACKBONE_DIR, exist_ok=True)
LOCAL_TEMP = "/local_disk0/tmp/sam3_backbone"

class BackboneFeatureHook:
    """Captures backbone output: [1, 1024, 72, 72]"""
    def __init__(self, model):
        self.captured = None
        self.handle = model.vision_encoder.backbone.register_forward_hook(self._hook_fn)
        print("  ✓ Hook on vision_encoder.backbone")
    
    def _hook_fn(self, module, input, output):
        if hasattr(output, 'last_hidden_state'):
            # last_hidden_state is [1, 5184, 1024] — reshape to spatial
            lhs = output.last_hidden_state  # [1, 5184, 1024]
            B, N, C = lhs.shape
            H = W = int(N ** 0.5)  # 72
            self.captured = lhs.permute(0, 2, 1).reshape(B, C, H, W).detach().cpu().half()
        elif isinstance(output, torch.Tensor):
            self.captured = output.detach().cpu().half()
    
    def get_and_clear(self):
        feat = self.captured
        self.captured = None
        return feat
    
    def remove(self):
        self.handle.remove()


# Check if already extracted
need_extract = []
for vid_path in BENCHMARK_VIDEOS:
    vid_id = osp.basename(osp.dirname(vid_path))
    out_dir = osp.join(BACKBONE_DIR, vid_id)
    if osp.exists(out_dir):
        count = len([f for f in os.listdir(out_dir) if f.endswith('.pt')])
        if count >= 500:
            print(f"  ✓ {vid_id}: {count} files (skip)")
            continue
    need_extract.append(vid_path)

if need_extract:
    print(f"\nExtracting backbone features for {len(need_extract)} videos...")
    hook = BackboneFeatureHook(model)
    
    for vid_path in need_extract:
        vid_id = osp.basename(osp.dirname(vid_path))
        frames_dir = prepare_frames_or_path(vid_path)
        local_dir = osp.join(LOCAL_TEMP, vid_id)
        dbfs_dir = osp.join(BACKBONE_DIR, vid_id)
        
        if osp.exists(local_dir): shutil.rmtree(local_dir)
        os.makedirs(local_dir, exist_ok=True)
        os.makedirs(dbfs_dir, exist_ok=True)
        
        prompts = load_bounding_box_prompts(get_prompt_path_from_video_path(vid_path))
        obj_names = sorted(prompts.keys())
        input_boxes = prepare_sam3_bbox_format([prompts[n] for n in obj_names])
        
        frame_files = sorted([osp.join(frames_dir, f) for f in os.listdir(frames_dir)
                              if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        
        session = processor.init_video_session(inference_device=device, dtype=DEFAULT_DTYPE)
        
        t0 = time.time()
        for fi, fp in enumerate(frame_files):
            frame = Image.open(fp).convert("RGB")
            inputs = processor(images=frame, device=device, return_tensors="pt")
            
            if fi == 0:
                processor.add_inputs_to_inference_session(
                    inference_session=session, frame_idx=0,
                    obj_ids=list(range(len(obj_names))), input_boxes=input_boxes,
                    original_size=inputs.original_sizes[0])
            
            with torch.no_grad():
                model(inference_session=session, frame=inputs.pixel_values[0])
            
            feat = hook.get_and_clear()
            if feat is not None:
                torch.save(feat, osp.join(local_dir, f"backbone_{fi:06d}.pt"))
            
            del inputs, frame
            if (fi + 1) % 100 == 0:
                gc.collect(); torch.cuda.empty_cache()
                print(f"    {vid_id}: [{fi+1}/{len(frame_files)}] {(fi+1)/(time.time()-t0):.1f} FPS")
        
        # Copy to DBFS
        os.system(f"cp -r '{local_dir}/'* '{dbfs_dir}/'")
        count = len([f for f in os.listdir(dbfs_dir) if f.endswith('.pt')])
        print(f"  ✓ {vid_id}: {count} backbone features saved")
        
        shutil.rmtree(local_dir)
        del session; gc.collect(); torch.cuda.empty_cache()
    
    hook.remove()
else:
    print("\n✓ All backbone features already extracted")

# Verify shape
sample_path = sorted(glob.glob(osp.join(BACKBONE_DIR, TRAIN_VIDEO_IDS[0], "backbone_*.pt")))[0]
sample = torch.load(sample_path, map_location='cpu')
print(f"\nBackbone feature shape: {list(sample.shape)}")
BACKBONE_SHAPE = list(sample.shape)
assert BACKBONE_SHAPE[-2:] == [72, 72], f"Expected 72x72, got {BACKBONE_SHAPE}"
assert BACKBONE_SHAPE[1] == 1024 or BACKBONE_SHAPE[-1] == 1024, f"Expected 1024 channels"
del sample

  ✓ 2019_11_05_000002: 600 files (skip)
  ✓ 2019_11_11_000028: 600 files (skip)
  ✓ 2019_11_11_000036: 600 files (skip)
  ✓ 2019_11_22_000010: 600 files (skip)
  ✓ 2019_11_28_000113: 600 files (skip)
  ✓ 2019_12_02_000005: 600 files (skip)
  ✓ 2019_12_02_000208: 600 files (skip)
  ✓ 2019_12_10_000060: 600 files (skip)
  ✓ 2019_12_10_000078: 600 files (skip)

✓ All backbone features already extracted

Backbone feature shape: [1, 1024, 72, 72]


Cell 7: Student Encoder Architecture (TinyViT-21M-512 + GroupNorm)


In [0]:
# CELL 7: STUDENT ENCODER — TinyViT-21M-512 + Multi-Scale FPN + GroupNorm
# =========================================================================
# All BatchNorm in TinyViT backbone is replaced with GroupNorm at init.
# This eliminates train/eval mismatch entirely — no eval() hacks needed.
# =========================================================================

def replace_bn_with_gn(module):
    """Recursively replace all BatchNorm layers with GroupNorm."""
    for name, child in module.named_children():
        if isinstance(child, (nn.BatchNorm2d, nn.BatchNorm1d)):
            nc = child.num_features
            # Find largest divisor of nc that is <= 32
            ng = min(32, nc)
            while nc % ng != 0:
                ng -= 1
            gn = nn.GroupNorm(ng, nc, affine=child.affine)
            if child.affine and child.weight is not None:
                gn.weight.data.copy_(child.weight.data)
                gn.bias.data.copy_(child.bias.data)
            setattr(module, name, gn)
        else:
            replace_bn_with_gn(child)


class StudentEncoderTinyViT(nn.Module):
    """
    TinyViT-21M-512 with multi-scale feature fusion.
    ALL normalization is GroupNorm — zero train/eval mismatch.
    """
    def __init__(self, out_channels=1024, pretrained=True):
        super().__init__()
        self.out_channels = out_channels

        self.backbone = timm.create_model(
            'tiny_vit_21m_512',
            pretrained=pretrained,
            features_only=True,
            out_indices=(1, 2, 3),
        )
        # Replace ALL BatchNorm in backbone with GroupNorm
        replace_bn_with_gn(self.backbone)

        # Per-stage lateral convs (already GroupNorm)
        self.lateral_s1 = nn.Sequential(
            nn.Conv2d(192, 256, 1, bias=False),
            nn.GroupNorm(32, 256),
            nn.GELU(),
        )
        self.lateral_s2 = nn.Sequential(
            nn.Conv2d(384, 256, 1, bias=False),
            nn.GroupNorm(32, 256),
            nn.GELU(),
        )
        self.lateral_s3 = nn.Sequential(
            nn.Conv2d(576, 256, 1, bias=False),
            nn.GroupNorm(32, 256),
            nn.GELU(),
        )

        # Project fused 768ch -> 1024
        self.channel_proj = nn.Sequential(
            nn.Conv2d(768, out_channels, 1, bias=False),
            nn.GroupNorm(32, out_channels),
            nn.GELU(),
        )

        # Spatial refinement at 72x72 — residual path
        self.spatial_refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(32, out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(32, out_channels),
        )

        # Learnable per-channel affine
        self.output_scale = nn.Parameter(torch.ones(1, out_channels, 1, 1))
        self.output_bias  = nn.Parameter(torch.zeros(1, out_channels, 1, 1))

        self._init_weights()

    def _init_weights(self):
        for module_group in [self.lateral_s1, self.lateral_s2, self.lateral_s3,
                             self.channel_proj, self.spatial_refine]:
            for layer in module_group.modules():
                if isinstance(layer, nn.Conv2d):
                    nn.init.kaiming_normal_(layer.weight, mode='fan_out')
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)
                elif isinstance(layer, nn.GroupNorm):
                    nn.init.ones_(layer.weight)
                    nn.init.zeros_(layer.bias)
        # Zero-init last GroupNorm so residual starts as identity
        nn.init.zeros_(self.spatial_refine[-1].weight)

    def forward(self, x):
        stages = self.backbone(x)

        TARGET_H, TARGET_W = 72, 72

        f1 = self.lateral_s1(stages[0])
        f2 = self.lateral_s2(stages[1])
        f3 = self.lateral_s3(stages[2])

        f1 = F.interpolate(f1, size=(TARGET_H, TARGET_W), mode='area')
        f2 = F.interpolate(f2, size=(TARGET_H, TARGET_W), mode='bilinear', align_corners=False)
        f3 = F.interpolate(f3, size=(TARGET_H, TARGET_W), mode='bilinear', align_corners=False)

        fused = torch.cat([f1, f2, f3], dim=1)

        out = self.channel_proj(fused)
        out = out + self.spatial_refine(out)
        out = out * self.output_scale + self.output_bias

        return out

    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters())


# ── Test ──────────────────────────────────────────────────────────
student = StudentEncoderTinyViT(out_channels=1024, pretrained=True).to(device)

# Verify NO BatchNorm remains anywhere
bn_count = sum(1 for m in student.modules() if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)))
gn_count = sum(1 for m in student.modules() if isinstance(m, nn.GroupNorm))
print(f"BatchNorm layers: {bn_count} (should be 0)")
print(f"GroupNorm layers: {gn_count}")

# Verify train/eval identical
student.train()
with torch.no_grad():
    dummy = torch.randn(1, 3, 1024, 1024).to(device)
    out_train = student(dummy)

student.eval()
with torch.no_grad():
    out_eval = student(dummy)

diff = (out_train - out_eval).abs()
print(f"Train vs Eval max diff: {diff.max():.8f} (should be 0.0)")
print(f"Output shape: {list(out_eval.shape)} {'OK' if list(out_eval.shape) == [1,1024,72,72] else 'WRONG'}")
print(f"Total params: {student.get_num_parameters()/1e6:.2f}M")

del dummy, out_train, out_eval, diff
gc.collect(); torch.cuda.empty_cache()

BatchNorm layers: 0 (should be 0)
GroupNorm layers: 33
Train vs Eval max diff: 2.26131129 (should be 0.0)
Output shape: [1, 1024, 72, 72] OK
Total params: 40.66M


Cell 8: Dataset + Augmentation


In [0]:
# CELL 8: DATASET + AUGMENTATION
# =========================================================================
# Changes:
#   1. TinyViT-512 normalization (not ImageNet default)
#   2. Horizontal flip augmentation with matched feature flipping
#   3. Shared STUDENT_TRANSFORM for use in verification + benchmark cells
# =========================================================================

BACKBONE_DIR = osp.join(BASE_DIR, "Backbone_Features")

# Get TinyViT-512 normalization config
_tmp = timm.create_model('tiny_vit_21m_512', pretrained=False)
_data_cfg = resolve_data_config(_tmp.pretrained_cfg)
del _tmp
TINYVIT_MEAN = _data_cfg['mean']
TINYVIT_STD  = _data_cfg['std']
print(f"TinyViT-512 normalization: mean={TINYVIT_MEAN}, std={TINYVIT_STD}")

# Shared transform (no augmentation) — used by Cell 11 verification + Cell 13 benchmark
STUDENT_TRANSFORM = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(mean=TINYVIT_MEAN, std=TINYVIT_STD),
])


class BackboneFeatureDataset(Dataset):
    """
    Paired dataset: RGB frame + pre-extracted backbone feature [1024, 72, 72].
    Supports synchronized horizontal flip augmentation.
    """
    def __init__(self, frames_base_dir, backbone_dir, video_ids,
                 image_size=(1024, 1024), augment=False):
        self.image_size = image_size
        self.augment = augment

        # Base transform (no flip — flip is applied manually for sync)
        self.base_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=TINYVIT_MEAN, std=TINYVIT_STD),
        ])
        # Color jitter (image only — teacher features already encode original colors)
        self.color_jitter = transforms.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02
        ) if augment else None

        self.pairs = []
        for vid in video_ids:
            fdir = osp.join(frames_base_dir, vid, "video1")
            bdir = osp.join(backbone_dir, vid)
            if not osp.exists(fdir) or not osp.exists(bdir):
                print(f"  Warning: Missing: {vid}")
                continue
            for bp in sorted(glob.glob(osp.join(bdir, "backbone_*.pt"))):
                fidx = int(osp.basename(bp).split('_')[1].split('.')[0])
                frame_path = osp.join(fdir, f"{fidx + 1:07d}.jpg")
                if osp.exists(frame_path):
                    self.pairs.append((frame_path, bp))
        print(f"Dataset: {len(self.pairs)} pairs from {len(video_ids)} videos"
              f" (augment={augment})")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        fp, bp = self.pairs[idx]
        img = Image.open(fp).convert('RGB')
        feat = torch.load(bp, map_location='cpu').float()
        if feat.dim() == 4:
            feat = feat.squeeze(0)  # [1024, 72, 72]

        # --- Augmentation (training only) ---
        if self.augment:
            # Color jitter on PIL image (does NOT affect teacher features)
            if self.color_jitter is not None:
                img = self.color_jitter(img)

            # Synchronized horizontal flip (50% chance)
            if torch.rand(1).item() > 0.5:
                img = transforms.functional.hflip(img)
                feat = feat.flip(-1)  # Flip width dimension of [C, H, W]

        img = self.base_transform(img)
        return img, feat


train_dataset = BackboneFeatureDataset(
    FRAMES_DIR, BACKBONE_DIR, TRAIN_VIDEO_IDS, augment=True)
val_dataset = BackboneFeatureDataset(
    FRAMES_DIR, BACKBONE_DIR, VAL_VIDEO_IDS, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
imgs, feats = next(iter(train_loader))
print(f"Batch: images={imgs.shape}, backbone_feat={feats.shape}")
print(f"  feat mean={feats.mean():.4f} std={feats.std():.4f}")
del imgs, feats



TinyViT-512 normalization: mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
Dataset: 2400 pairs from 4 videos (augment=True)
Dataset: 600 pairs from 1 videos (augment=False)
Train: 2400 | Val: 600
Batch: images=torch.Size([4, 3, 1024, 1024]), backbone_feat=torch.Size([4, 1024, 72, 72])
  feat mean=0.0064 std=1.4390


Cell 9: Loss + Training Components (Rebalanced)


In [0]:
# CELL 9: LOSS + TRAINING COMPONENTS
# =========================================================================
# Changes:
#   1. Directional MSE on L2-normalized features (learns direction first)
#   2. scale_w reduced to 0.3 (prevents scale dominating over direction)
#   3. EMA decay 0.9995 (was 0.999 — too fast for ~500 steps/epoch)
#   4. CosineAnnealingWarmRestarts(T_0=15) — completes cycles before early stop
#   5. output_scale/output_bias at 3e-4 (same as projection, not 100x higher)
#   6. Per-param-group setup uses new StudentEncoderTinyViT attributes
# =========================================================================

class BackboneDistillationLoss(nn.Module):
    """
    Feature distillation loss with separate direction + scale learning.
    Direction (normalized MSE + cosine) is prioritized over scale matching.
    """
    def __init__(self, dir_w=1.0, cos_w=0.5, scale_w=0.3, raw_mse_w=0.1):
        super().__init__()
        self.dir_w = dir_w
        self.cos_w = cos_w
        self.scale_w = scale_w
        self.raw_mse_w = raw_mse_w

    def forward(self, pred, target):
        B, C, H, W = pred.shape

        # 1. Directional MSE on L2-normalized features (direction only)
        pred_norm = F.normalize(pred, p=2, dim=1)
        target_norm = F.normalize(target, p=2, dim=1)
        mse_dir = F.mse_loss(pred_norm, target_norm)

        # 2. Cosine loss on spatial patches
        pf = pred.reshape(B, C, -1).permute(0, 2, 1).reshape(-1, C)
        tf = target.reshape(B, C, -1).permute(0, 2, 1).reshape(-1, C)
        cosine = F.cosine_embedding_loss(
            pf, tf, torch.ones(pf.size(0), device=pred.device))

        # 3. Channel-wise scale matching (per-channel std + mean)
        pred_ch_std  = pred.std(dim=(2, 3))    # [B, C]
        target_ch_std = target.std(dim=(2, 3))
        pred_ch_mean = pred.mean(dim=(2, 3))
        target_ch_mean = target.mean(dim=(2, 3))
        scale = (F.mse_loss(pred_ch_std, target_ch_std) +
                 F.mse_loss(pred_ch_mean, target_ch_mean))

        # 4. Raw MSE (auxiliary, low weight)
        mse_raw = F.mse_loss(pred, target)

        total = (self.dir_w * mse_dir + self.cos_w * cosine +
                 self.scale_w * scale + self.raw_mse_w * mse_raw)

        details = {
            'total_loss': total.item(),
            'mse_dir': mse_dir.item(),
            'cosine': cosine.item(),
            'scale': scale.item(),
            'mse_raw': mse_raw.item(),
            'pred_std': pred.std().item(),
            'target_std': target.std().item(),
            'scale_ratio': pred.std().item() / (target.std().item() + 1e-8),
        }
        return total, details


class ModelEMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.shadow = {n: p.data.clone()
                       for n, p in model.named_parameters() if p.requires_grad}

    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n] = self.decay * self.shadow[n] + (1 - self.decay) * p.data

    def apply_shadow(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data.copy_(self.backup[n])
        self.backup = {}


class EarlyStopping:
    def __init__(self, patience=15, min_delta=1e-5):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_loss, self.best_epoch, self.early_stop = 0, None, 0, False

    def __call__(self, val_loss, epoch):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.best_epoch, self.counter = val_loss, epoch, 0
        else:
            self.counter += 1
            print(f"  EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# ── Initialize ────────────────────────────────────────────────────────────
criterion = BackboneDistillationLoss(
    dir_w=1.0, cos_w=0.5, scale_w=0.3, raw_mse_w=0.1
)

# All projection/affine params at same lr — no 100x shortcut for output_scale
optimizer = AdamW([
    {'params': student.backbone.parameters(),       'lr': 1e-4,  'weight_decay': 1e-4},
    {'params': student.lateral_s1.parameters(),     'lr': 3e-4,  'weight_decay': 1e-4},
    {'params': student.lateral_s2.parameters(),     'lr': 3e-4,  'weight_decay': 1e-4},
    {'params': student.lateral_s3.parameters(),     'lr': 3e-4,  'weight_decay': 1e-4},
    {'params': student.channel_proj.parameters(),   'lr': 3e-4,  'weight_decay': 1e-4},
    {'params': student.spatial_refine.parameters(), 'lr': 3e-4,  'weight_decay': 1e-4},
    {'params': [student.output_scale,
                student.output_bias],                'lr': 3e-4,  'weight_decay': 0.0},
], lr=3e-4)

# CosineAnnealingWarmRestarts: T_0=15 epochs per cycle, completes 1+ cycles
# before early stopping (patience=15) can fire
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-6)

scaler = GradScaler(enabled=True)
ema = ModelEMA(student, decay=0.9995)
early_stopping = EarlyStopping(patience=15)

print("Training components ready:")
print(f"  Loss: dir_w=1.0, cos_w=0.5, scale_w=0.3, raw_mse_w=0.1")
print(f"  EMA decay: 0.9995")
print(f"  Scheduler: CosineAnnealingWarmRestarts(T_0=15, T_mult=2)")
print(f"  EarlyStopping patience: 15")
print(f"  output_scale/bias lr: 3e-4 (same as projection)")



Training components ready:
  Loss: dir_w=1.0, cos_w=0.5, scale_w=0.3, raw_mse_w=0.1
  EMA decay: 0.9995
  Scheduler: CosineAnnealingWarmRestarts(T_0=15, T_mult=2)
  EarlyStopping patience: 15
  output_scale/bias lr: 3e-4 (same as projection)


/root/.ipykernel/4583/command-7767092744230259-452668610:127: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=True)


Cell 10: Training Loop


In [0]:
# CELL 10: TRAINING LOOP (BACKBONE DISTILLATION)
# =========================================================================
# Changes:
#   1. scheduler.step() per epoch (not per batch) for warm restarts
#   2. Logging keys match new loss (mse_dir, cosine, scale, mse_raw)
#   3. Logs scale_ratio for monitoring convergence
# =========================================================================

CHECKPOINT_DIR = osp.join(BASE_DIR, "Checkpoints_Backbone")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"\n{'#'*70}")
print(f"BACKBONE DISTILLATION: StudentEncoderTinyViT -> [1024, 72, 72]")
print(f"{'#'*70}")
print(f"Student: {student.get_num_parameters()/1e6:.2f}M params")
print(f"Target:  SAM3 backbone output [1024, 72, 72]")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Batch: 4")
print(f"{'#'*70}\n")

best_val_loss = float('inf')
history = []

for epoch in range(100):
    # ── Train ─────────────────────────────────────────────────────────
    student.train()
    run = defaultdict(float)

    for batch_idx, (images, targets) in enumerate(train_loader):
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()

        with autocast(enabled=True):
            pred = student(images)
            loss, ld = criterion(pred, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        ema.update(student)

        for k, v in ld.items():
            run[k] += v

        if (batch_idx + 1) % 20 == 0:
            n = batch_idx + 1
            print(f"  [E{epoch+1}] B{n}/{len(train_loader)} "
                  f"Loss:{run['total_loss']/n:.4f} "
                  f"DirMSE:{run['mse_dir']/n:.4f} "
                  f"Cos:{run['cosine']/n:.4f} "
                  f"ScaleR:{run['scale_ratio']/n:.3f}")

        del images, targets, pred, loss

    train_loss = run['total_loss'] / len(train_loader)

    # Step scheduler once per epoch (warm restarts)
    scheduler.step(epoch)

    # ── Validate ──────────────────────────────────────────────────────
    student.eval()
    vrun = defaultdict(float)
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            with autocast(enabled=True):
                pred = student(images)
                loss, ld = criterion(pred, targets)
            for k, v in ld.items():
                vrun[k] += v
            del images, targets, pred, loss
    val_loss = vrun['total_loss'] / len(val_loader)

    # ── EMA validate ──────────────────────────────────────────────────
    ema.apply_shadow(student)
    ema_val = 0.0
    ema_cos = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            with autocast(enabled=True):
                pred = student(images)
                loss, ld_ema = criterion(pred, targets)
            ema_val += loss.item()
            ema_cos += ld_ema['cosine']
            del images, targets, pred, loss
    ema_val /= len(val_loader)
    ema_cos /= len(val_loader)
    ema.restore(student)

    # ── Log ───────────────────────────────────────────────────────────
    epoch_log = {
        'epoch': epoch + 1, 'train_loss': train_loss,
        'val_loss': val_loss, 'ema_val_loss': ema_val,
        'val_mse_dir': vrun['mse_dir'] / len(val_loader),
        'val_cosine': vrun['cosine'] / len(val_loader),
        'val_scale': vrun['scale'] / len(val_loader),
        'val_mse_raw': vrun['mse_raw'] / len(val_loader),
        'val_pred_std': vrun['pred_std'] / len(val_loader),
        'val_target_std': vrun['target_std'] / len(val_loader),
        'val_scale_ratio': vrun['scale_ratio'] / len(val_loader),
        'ema_cosine': ema_cos,
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(epoch_log)

    print(f"\n  Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f} "
          f"ema_val={ema_val:.4f}")
    print(f"    DirMSE={epoch_log['val_mse_dir']:.4f} "
          f"Cosine={epoch_log['val_cosine']:.4f} "
          f"ScaleRatio={epoch_log['val_scale_ratio']:.3f} "
          f"EMA_Cos={ema_cos:.4f} "
          f"lr={epoch_log['lr']:.2e}")

    # ── Checkpoint ────────────────────────────────────────────────────
    if ema_val < best_val_loss:
        best_val_loss = ema_val
        print(f"    New best: {best_val_loss:.4f}")
        torch.save({
            'epoch': epoch,
            'model_state_dict': student.state_dict(),
            'ema_shadow': ema.shadow,
            'optimizer': optimizer.state_dict(),
            'metrics': epoch_log,
        }, osp.join(CHECKPOINT_DIR, "best_backbone_student.pt"))

    if (epoch + 1) % 10 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': student.state_dict()},
                   osp.join(CHECKPOINT_DIR, f"backbone_student_epoch_{epoch+1}.pt"))

    if early_stopping(ema_val, epoch):
        print(f"\n  EARLY STOPPING at epoch {epoch+1}")
        break

    gc.collect(); torch.cuda.empty_cache()
    print()

with open(osp.join(CHECKPOINT_DIR, "backbone_distill_history.json"), 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n{'#'*70}")
print(f"DONE: Best EMA val loss = {best_val_loss:.4f}")
print(f"{'#'*70}")




######################################################################
BACKBONE DISTILLATION: StudentEncoderTinyViT -> [1024, 72, 72]
######################################################################
Student: 40.66M params
Target:  SAM3 backbone output [1024, 72, 72]
Train: 2400 | Val: 600 | Batch: 4
######################################################################



/root/.ipykernel/2127/command-7767092744230261-20602163:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):


  [E1] B20/600 Loss:1.1325 DirMSE:0.0016 Cos:0.8298 ScaleR:0.367
  [E1] B40/600 Loss:1.0431 DirMSE:0.0015 Cos:0.7711 ScaleR:0.382
  [E1] B60/600 Loss:0.9681 DirMSE:0.0014 Cos:0.7190 ScaleR:0.404
  [E1] B80/600 Loss:0.9154 DirMSE:0.0013 Cos:0.6779 ScaleR:0.426
  [E1] B100/600 Loss:0.8764 DirMSE:0.0013 Cos:0.6443 ScaleR:0.442
  [E1] B120/600 Loss:0.8465 DirMSE:0.0012 Cos:0.6172 ScaleR:0.454
  [E1] B140/600 Loss:0.8229 DirMSE:0.0012 Cos:0.5948 ScaleR:0.464
  [E1] B160/600 Loss:0.8034 DirMSE:0.0011 Cos:0.5761 ScaleR:0.471
  [E1] B180/600 Loss:0.7872 DirMSE:0.0011 Cos:0.5602 ScaleR:0.478
  [E1] B200/600 Loss:0.7732 DirMSE:0.0011 Cos:0.5466 ScaleR:0.484


/root/.ipykernel/2127/command-7767092744230261-20602163:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):


  [E1] B220/600 Loss:0.7610 DirMSE:0.0010 Cos:0.5345 ScaleR:0.489
  [E1] B240/600 Loss:0.7500 DirMSE:0.0010 Cos:0.5237 ScaleR:0.493
  [E1] B260/600 Loss:0.7402 DirMSE:0.0010 Cos:0.5140 ScaleR:0.497
  [E1] B280/600 Loss:0.7313 DirMSE:0.0010 Cos:0.5054 ScaleR:0.501
  [E1] B300/600 Loss:0.7232 DirMSE:0.0010 Cos:0.4976 ScaleR:0.504
  [E1] B320/600 Loss:0.7155 DirMSE:0.0010 Cos:0.4903 ScaleR:0.507
  [E1] B340/600 Loss:0.7085 DirMSE:0.0009 Cos:0.4836 ScaleR:0.510
  [E1] B360/600 Loss:0.7018 DirMSE:0.0009 Cos:0.4774 ScaleR:0.513
  [E1] B380/600 Loss:0.6956 DirMSE:0.0009 Cos:0.4717 ScaleR:0.516
  [E1] B400/600 Loss:0.6898 DirMSE:0.0009 Cos:0.4664 ScaleR:0.518
  [E1] B420/600 Loss:0.6842 DirMSE:0.0009 Cos:0.4613 ScaleR:0.521
  [E1] B440/600 Loss:0.6789 DirMSE:0.0009 Cos:0.4566 ScaleR:0.523
  [E1] B460/600 Loss:0.6739 DirMSE:0.0009 Cos:0.4522 ScaleR:0.525
  [E1] B480/600 Loss:0.6690 DirMSE:0.0009 Cos:0.4480 ScaleR:0.527
  [E1] B500/600 Loss:0.6643 DirMSE:0.0009 Cos:0.4441 ScaleR:0.529
  [E1] B52

...

  [E36] B20/600 Loss:0.1820 DirMSE:0.0004 Cos:0.1854 ScaleR:0.945
  [E36] B40/600 Loss:0.1815 DirMSE:0.0004 Cos:0.1851 ScaleR:0.945
  [E36] B60/600 Loss:0.1809 DirMSE:0.0004 Cos:0.1844 ScaleR:0.945
  [E36] B80/600 Loss:0.1811 DirMSE:0.0004 Cos:0.1846 ScaleR:0.945
  [E36] B100/600 Loss:0.1807 DirMSE:0.0004 Cos:0.1842 ScaleR:0.945
  [E36] B120/600 Loss:0.1806 DirMSE:0.0004 Cos:0.1843 ScaleR:0.945
  [E36] B140/600 Loss:0.1806 DirMSE:0.0004 Cos:0.1842 ScaleR:0.945
  [E36] B160/600 Loss:0.1808 DirMSE:0.0004 Cos:0.1844 ScaleR:0.945
  [E36] B180/600 Loss:0.1808 DirMSE:0.0004 Cos:0.1844 ScaleR:0.945
  [E36] B200/600 Loss:0.1808 DirMSE:0.0004 Cos:0.1843 ScaleR:0.945
  [E36] B220/600 Loss:0.1808 DirMSE:0.0004 Cos:0.1843 ScaleR:0.945
  [E36] B240/600 Loss:0.1807 DirMSE:0.0004 Cos:0.1842 ScaleR:0.945
  [E36] B260/600 Loss:0.1806 DirMSE:0.0004 Cos:0.1841 ScaleR:0.945
  [E36] B280/600 Loss:0.1807 DirMSE:0.0004 Cos:0.1841 ScaleR:0.945
  [E36] B300/600 Loss:0.1807 DirMSE:0.0004 Cos:0.1841 ScaleR:0.945

/root/.ipykernel/2127/command-7767092744230261-20602163:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):
/root/.ipykernel/2127/command-7767092744230261-20602163:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):



  Epoch 37: train=0.1800 val=0.2597 ema_val=0.2578
    DirMSE=0.0005 Cosine=0.2565 ScaleRatio=0.951 EMA_Cos=0.2556 lr=2.14e-05
    New best: 0.2578



/root/.ipykernel/2127/command-7767092744230261-20602163:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):


  [E38] B20/600 Loss:0.1788 DirMSE:0.0004 Cos:0.1831 ScaleR:0.945
  [E38] B40/600 Loss:0.1794 DirMSE:0.0004 Cos:0.1834 ScaleR:0.945
  [E38] B60/600 Loss:0.1795 DirMSE:0.0004 Cos:0.1837 ScaleR:0.945
  [E38] B80/600 Loss:0.1798 DirMSE:0.0004 Cos:0.1839 ScaleR:0.945
  [E38] B100/600 Loss:0.1800 DirMSE:0.0004 Cos:0.1839 ScaleR:0.945
  [E38] B120/600 Loss:0.1797 DirMSE:0.0004 Cos:0.1835 ScaleR:0.945
  [E38] B140/600 Loss:0.1798 DirMSE:0.0004 Cos:0.1835 ScaleR:0.946
  [E38] B160/600 Loss:0.1798 DirMSE:0.0004 Cos:0.1835 ScaleR:0.946
  [E38] B180/600 Loss:0.1799 DirMSE:0.0004 Cos:0.1836 ScaleR:0.946
  [E38] B200/600 Loss:0.1797 DirMSE:0.0004 Cos:0.1834 ScaleR:0.946
  [E38] B220/600 Loss:0.1798 DirMSE:0.0004 Cos:0.1835 ScaleR:0.946
  [E38] B240/600 Loss:0.1796 DirMSE:0.0004 Cos:0.1833 ScaleR:0.946
  [E38] B260/600 Loss:0.1796 DirMSE:0.0004 Cos:0.1834 ScaleR:0.946
  [E38] B280/600 Loss:0.1796 DirMSE:0.0004 Cos:0.1834 ScaleR:0.946
  [E38] B300/600 Loss:0.1797 DirMSE:0.0004 Cos:0.1834 ScaleR:0.946

Cell 11: Verify Feature Quality (Raw Diagnostic)


In [0]:
# CELL 11: LOAD BEST STUDENT + VERIFY FEATURE QUALITY (RAW — NO CALIBRATION)
# =========================================================================
# Changes:
#   1. Uses StudentEncoderTinyViT (not StudentEncoderBackbone)
#   2. Uses shared STUDENT_TRANSFORM (TinyViT normalization)
#   3. NO post-hoc calibration — raw metrics are the true diagnostic
#   4. Train vs eval consistency check (should be ~0 diff with GroupNorm)
# =========================================================================
CHECKPOINT_DIR = osp.join(BASE_DIR, "Checkpoints_Backbone")

ckpt = torch.load(osp.join(CHECKPOINT_DIR, "best_backbone_student.pt"),
                   map_location=device)

student_final = StudentEncoderTinyViT(out_channels=1024, pretrained=False).to(device)
if 'ema_shadow' in ckpt:
    for name, param in student_final.named_parameters():
        if name in ckpt['ema_shadow']:
            param.data.copy_(ckpt['ema_shadow'][name])
    print(f"Loaded EMA weights (epoch {ckpt['epoch']+1})")
else:
    student_final.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded model weights (epoch {ckpt['epoch']+1})")

# ── Train vs Eval consistency (GroupNorm = no difference) ─────────
sample_backbone_files = sorted(glob.glob(
    osp.join(BACKBONE_DIR, TRAIN_VIDEO_IDS[0], "backbone_*.pt")))

bp0 = sample_backbone_files[0]
fidx0 = int(osp.basename(bp0).split('_')[1].split('.')[0])
frame_path0 = osp.join(FRAMES_DIR, TRAIN_VIDEO_IDS[0], "video1", f"{fidx0+1:07d}.jpg")
test_img = STUDENT_TRANSFORM(Image.open(frame_path0).convert("RGB")).unsqueeze(0).to(device)

student_final.train()
with torch.no_grad():
    out_train = student_final(test_img)

student_final.eval()
with torch.no_grad():
    out_eval = student_final(test_img)

diff = (out_train - out_eval).abs()
print(f"\nTrain vs Eval max diff:  {diff.max():.8f}")
print(f"Train vs Eval mean diff: {diff.mean():.8f}")
if diff.max() < 1e-5:
    print("  GroupNorm verified: train/eval identical")
else:
    print("  WARNING: train/eval mismatch detected!")
del test_img, out_train, out_eval, diff

# ── Feature quality verification (RAW — no calibration) ──────────
student_final.eval()
print(f"\nVerifying on {min(20, len(sample_backbone_files))} sample frames (RAW, no calibration)...")

mse_list, cos_list, ratio_list = [], [], []
N_SAMPLES = min(20, len(sample_backbone_files))

for i in range(N_SAMPLES):
    bp = sample_backbone_files[i]
    fidx = int(osp.basename(bp).split('_')[1].split('.')[0])
    frame_path = osp.join(FRAMES_DIR, TRAIN_VIDEO_IDS[0], "video1",
                          f"{fidx + 1:07d}.jpg")

    teacher_feat = torch.load(bp, map_location='cpu').float()
    if teacher_feat.dim() == 4:
        teacher_feat = teacher_feat.squeeze(0)

    frame_tensor = STUDENT_TRANSFORM(
        Image.open(frame_path).convert("RGB")
    ).unsqueeze(0).to(device)
    with torch.no_grad():
        student_feat = student_final(frame_tensor).cpu().squeeze(0)

    mse = F.mse_loss(student_feat, teacher_feat).item()
    cos = F.cosine_similarity(
        student_feat.reshape(1, -1), teacher_feat.reshape(1, -1)).item()
    ratio = student_feat.std().item() / teacher_feat.std().item()

    mse_list.append(mse)
    cos_list.append(cos)
    ratio_list.append(ratio)

    del frame_tensor, student_feat, teacher_feat

print(f"\n{'Metric':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print(f"{'---'*20}")
print(f"{'MSE':<20} {np.mean(mse_list):>10.4f} {np.std(mse_list):>10.4f} "
      f"{np.min(mse_list):>10.4f} {np.max(mse_list):>10.4f}")
print(f"{'Cosine':<20} {np.mean(cos_list):>10.4f} {np.std(cos_list):>10.4f} "
      f"{np.min(cos_list):>10.4f} {np.max(cos_list):>10.4f}")
print(f"{'Scale Ratio':<20} {np.mean(ratio_list):>10.4f} {np.std(ratio_list):>10.4f} "
      f"{np.min(ratio_list):>10.4f} {np.max(ratio_list):>10.4f}")
print(f"{'---'*20}")

avg_ratio = np.mean(ratio_list)
avg_cos = np.mean(cos_list)

# Diagnostic summary
print(f"\n--- Diagnostic ---")
if avg_cos >= 0.7:
    print(f"  Cosine {avg_cos:.4f}: GOOD — features directionally aligned")
elif avg_cos >= 0.5:
    print(f"  Cosine {avg_cos:.4f}: MODERATE — may need more epochs")
else:
    print(f"  Cosine {avg_cos:.4f}: LOW — check loss weights / architecture")

if 0.8 <= avg_ratio <= 1.2:
    print(f"  Scale ratio {avg_ratio:.3f}: GOOD")
elif 0.5 <= avg_ratio <= 2.0:
    print(f"  Scale ratio {avg_ratio:.3f}: MILD MISMATCH — acceptable")
else:
    print(f"  Scale ratio {avg_ratio:.3f}: SEVERE MISMATCH — check scale_w")

Loaded EMA weights (epoch 75)

Train vs Eval max diff:  63.35067749
Train vs Eval mean diff: 0.03444041

Verifying on 20 sample frames (RAW, no calibration)...

Metric                     Mean        Std        Min        Max
------------------------------------------------------------
MSE                      0.7631     0.0224     0.7286     0.8109
Cosine                   0.8080     0.0063     0.7949     0.8172
Scale Ratio              0.9489     0.0032     0.9415     0.9556
------------------------------------------------------------

--- Diagnostic ---
  Cosine 0.8080: GOOD — features directionally aligned
  Scale ratio 0.949: GOOD


Cell 12: PerformanceTracker + Evaluation

In [0]:
import motmetrics as mm

class PerformanceTracker:
    def __init__(self, model_name, model=None, log_dir=METRICS_DIR):
        self.model_name = model_name
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        self._frame_times_s = []
        self._video_start = self._frame_start = None
        self._gpu_peak = self._gpu_before = None
        self._throughput_fps = None
        self.total_frames = 0
        self._mask_areas = []
        self._tracking = {}
        self.model_size_metrics = {}
        if model:
            t = sum(p.numel() for p in model.parameters())
            self.model_size_metrics = {
                'total_parameters': t, 'total_parameters_millions': round(t/1e6, 2),
                'model_size_mb': round(t*2/1024**2, 2), 'model_size_gb': round(t*2/1024**3, 4),
            }
    
    def start_video_inference(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
            self._gpu_before = torch.cuda.memory_allocated()/1024**3
        self._video_start = time.time()
    
    def end_video_inference(self, n):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            self._gpu_peak = torch.cuda.max_memory_allocated()/1024**3
        self._throughput_fps = n / (time.time() - self._video_start)
        self.total_frames += n
    
    def start_frame(self):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        self._frame_start = time.time()
    
    def end_frame(self):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        if self._frame_start:
            self._frame_times_s.append(time.time() - self._frame_start)
            self._frame_start = None
    
    def start_timer(self):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        self._legacy = time.time()
    
    def end_timer(self):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        return time.time() - self._legacy
    
    def record_mask_quality(self, mask_area=0):
        if mask_area > 0: self._mask_areas.append(mask_area)
    
    def add_tracking_quality_metrics(self, s):
        pct = {'idf1','idp','idr','recall','precision','mota','motp'}
        for k, v in s.items():
            self._tracking[k] = float(v*100) if k in pct else (int(v) if isinstance(v,(int,float)) else v)
    
    def get_summary(self):
        fps = self._throughput_fps or 0
        ms = float(np.mean(self._frame_times_s)*1000) if self._frame_times_s else (1000/fps if fps else 0)
        return {
            "model_name": self.model_name, "total_frames": self.total_frames,
            "model_size_metrics": self.model_size_metrics,
            "performance_metrics": {
                "inference_time_ms_mean": ms,
                "inference_time_ms_std": float(np.std(self._frame_times_s)*1000) if self._frame_times_s else 0,
                "throughput_fps": round(fps, 4),
                "latency_ms_per_frame": round(1000/fps, 2) if fps else None,
                "gpu_memory_peak_gb": round(self._gpu_peak, 4) if self._gpu_peak else None,
                "gpu_memory_before_gb": round(self._gpu_before, 4) if self._gpu_before else None,
            },
            "mask_quality_metrics": {
                "mask_areas_mean": float(np.mean(self._mask_areas)) if self._mask_areas else 0,
                "mask_areas_std": float(np.std(self._mask_areas)) if self._mask_areas else 0,
            },
            "tracking_quality_metrics": self._tracking,
        }
    
    def save_metrics(self, video_id):
        fp = osp.join(self.log_dir, f"{self.model_name}_{video_id}_metrics.json")
        with open(fp, "w") as f:
            json.dump({"video_id": video_id, "summary": self.get_summary()}, f, indent=2)
        print(f"✓ Saved: {fp}")
        return fp
    
    def print_summary(self):
        s = self.get_summary()
        pm, tq, mm_s = s["performance_metrics"], s["tracking_quality_metrics"], s.get("model_size_metrics",{})
        mq = s["mask_quality_metrics"]
        print(f"\n{'='*70}")
        print(f"Summary: {self.model_name}")
        if mm_s: print(f"  Params: {mm_s.get('total_parameters_millions',0):.2f}M")
        print(f"  FPS: {pm['throughput_fps']:.2f} | Latency: {pm['inference_time_ms_mean']:.1f}ms")
        if pm.get('gpu_memory_peak_gb'): print(f"  GPU Peak: {pm['gpu_memory_peak_gb']:.3f} GB")
        if mq['mask_areas_mean'] > 0: print(f"  Mask Area: {mq['mask_areas_mean']:.0f} ± {mq['mask_areas_std']:.0f}")
        if tq: print(f"  MOTA: {tq.get('mota',0):.1f}% | IDF1: {tq.get('idf1',0):.1f}% | "
                      f"Precision: {tq.get('precision',0):.1f}% | Recall: {tq.get('recall',0):.1f}%")
        print(f"{'='*70}")


def evaluate_tracking_quality(video_id, track_dir, iou_threshold=0.5):
    date_part, seq_part = video_id[:10], video_id[-6:]
    gt_json = f"/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/annotated/{date_part}/{seq_part}/output.json"
    if not osp.exists(gt_json):
        print(f"⚠ GT not found: {gt_json}"); return {}
    
    with open(gt_json) as f: gt_data = json.load(f)
    
    gt_boxes = {}
    for obj in gt_data["objects"]:
        pid = int(obj["id"])
        for i, fr in enumerate(obj["frames"]):
            end = obj["frames"][i+1]["frameNumber"] if i+1 < len(obj["frames"]) else 600
            x, y, w, h = fr["bbox"].values()
            for fi in range(fr["frameNumber"], end):
                gt_boxes[(fi, pid)] = (x, y, x+w, y+h)
    
    tr_boxes = {}
    for fname in os.listdir(track_dir):
        if not fname.startswith("annotations_id") or not fname.endswith(".json"): continue
        pid = int(fname.split()[-1].split(".")[0])
        data = json.load(open(osp.join(track_dir, fname)))
        for frame_name, ann in data.items():
            fi = int(frame_name.split(".")[0]) - 1
            x, y, w, h = ann["bounding_box"]
            tr_boxes[(fi, pid)] = (x, y, x+w, y+h)
    
    acc = mm.MOTAccumulator(auto_id=True)
    for fi in range(600):
        gt_ids = [p for (f,p) in gt_boxes if f == fi]
        det_ids = [p for (f,p) in tr_boxes if f == fi]
        cost = np.zeros((len(gt_ids), len(det_ids)))
        for i, gid in enumerate(gt_ids):
            g = gt_boxes[(fi, gid)]
            for j, did in enumerate(det_ids):
                t = tr_boxes[(fi, did)]
                xa, ya = max(g[0],t[0]), max(g[1],t[1])
                xb, yb = min(g[2],t[2]), min(g[3],t[3])
                inter = max(0,xb-xa)*max(0,yb-ya)
                union = (g[2]-g[0])*(g[3]-g[1])+(t[2]-t[0])*(t[3]-t[1])-inter
                cost[i,j] = 1-(inter/union if union>0 else 0)
        cost[cost > 1-iou_threshold] = np.nan
        acc.update(gt_ids, det_ids, cost)
    
    mh = mm.metrics.create()
    summary = mh.compute(acc, metrics=mm.metrics.motchallenge_metrics, name=video_id)
    return {n: float(v) if isinstance(v,(float,np.floating)) else int(v) for n,v in summary.loc[video_id].items()}

print("✓ PerformanceTracker + evaluate_tracking_quality ready")

✓ PerformanceTracker + evaluate_tracking_quality ready


In [0]:
# CELL 12.5: INVESTIGATE SESSION MEMORY STRUCTURE
# =========================================================================
# Find exactly where the VRAM grows inside SAM3's inference session
# so we can prune it properly in Cell 13
# =========================================================================

test_vp = BENCHMARK_VIDEOS[0]
frames_dir = prepare_frames_or_path(test_vp)
prompts = load_bounding_box_prompts(get_prompt_path_from_video_path(test_vp))
obj_names = sorted(prompts.keys())
input_boxes = prepare_sam3_bbox_format([prompts[n] for n in obj_names])

frames = sorted([osp.join(frames_dir, f) for f in os.listdir(frames_dir)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:60]

session = processor.init_video_session(inference_device=device, dtype=DEFAULT_DTYPE)

# Snapshot VRAM at intervals
vram_snapshots = {}
for fi, fp in enumerate(frames):
    frame = Image.open(fp).convert("RGB")
    inputs = processor(images=frame, device=device, return_tensors="pt")
    if fi == 0:
        processor.add_inputs_to_inference_session(
            inference_session=session, frame_idx=0,
            obj_ids=list(range(len(obj_names))), input_boxes=input_boxes,
            original_size=inputs.original_sizes[0])
    with torch.no_grad():
        model(inference_session=session, frame=inputs.pixel_values[0])
    del inputs, frame
    if (fi + 1) in [1, 10, 20, 30, 40, 50, 60]:
        torch.cuda.synchronize()
        vram_snapshots[fi + 1] = torch.cuda.memory_allocated() / 1024**3

print("=== VRAM Growth ===")
for frame_n, gb in vram_snapshots.items():
    print(f"  Frame {frame_n:>3}: {gb:.3f} GB")
if len(vram_snapshots) >= 2:
    vals = list(vram_snapshots.values())
    frames_n = list(vram_snapshots.keys())
    growth_per_frame = (vals[-1] - vals[1]) / (frames_n[-1] - frames_n[1])
    print(f"  Growth rate: {growth_per_frame*1000:.2f} MB/frame")
    print(f"  Projected @ 600 frames: {vals[1] + growth_per_frame * 600:.2f} GB")
    print(f"  Projected @ 1800 frames (1min @30fps): {vals[1] + growth_per_frame * 1800:.2f} GB")

# ── Inspect session structure ─────────────────────────────────────────────
print("\n=== Session Attributes ===")
for attr in sorted(dir(session)):
    if attr.startswith('_') or callable(getattr(session, attr, None)):
        continue
    val = getattr(session, attr, None)
    if val is None:
        continue
    if isinstance(val, (int, float, bool, str)):
        print(f"  {attr} = {val}")
    elif isinstance(val, (list, tuple)):
        print(f"  {attr}: list, len={len(val)}")
    elif isinstance(val, dict):
        print(f"  {attr}: dict, len={len(val)}, keys={list(val.keys())[:5]}")
    elif hasattr(val, 'shape'):
        print(f"  {attr}: tensor {val.shape} {val.dtype}")

# ── Deep dive: output_dict_per_obj (main suspect) ────────────────────────
print("\n=== output_dict_per_obj Structure ===")
if hasattr(session, 'output_dict_per_obj'):
    odpo = session.output_dict_per_obj
    print(f"  Length: {len(odpo)} (one per object)")
    for obj_idx in range(min(2, len(odpo))):  # Just first 2 objects
        obj_dict = odpo[obj_idx]
        print(f"\n  Object {obj_idx}: keys = {list(obj_dict.keys())}")
        for key, sub_dict in obj_dict.items():
            if isinstance(sub_dict, dict):
                n_frames = len(sub_dict)
                print(f"    {key}: dict, {n_frames} frames stored")
                # Inspect one frame's data
                if n_frames > 0:
                    sample_frame_key = list(sub_dict.keys())[0]
                    sample = sub_dict[sample_frame_key]
                    if isinstance(sample, dict):
                        total_mb = 0
                        print(f"      Sample frame [{sample_frame_key}] keys:")
                        for k, v in sample.items():
                            if hasattr(v, 'shape'):
                                mb = v.element_size() * v.nelement() / 1e6
                                total_mb += mb
                                print(f"        {k}: {v.shape} {v.dtype} ({mb:.2f} MB)")
                            elif isinstance(v, (list, tuple)):
                                print(f"        {k}: {type(v).__name__}, len={len(v)}")
                            else:
                                print(f"        {k}: {type(v).__name__}")
                        print(f"      Per-frame total: {total_mb:.2f} MB")
                        print(f"      Projected for {n_frames} frames x 8 objects: "
                              f"{total_mb * n_frames * 8:.1f} MB")
            else:
                print(f"    {key}: {type(sub_dict).__name__}")

# ── Check other session dicts ─────────────────────────────────────────────
print("\n=== Other Session Dicts ===")
for attr in ['frames_tracked_per_obj', 'point_inputs_per_obj',
             'mask_inputs_per_obj', 'obj_with_new_inputs',
             'cached_features', 'recently_visited']:
    val = getattr(session, attr, None)
    if val is not None:
        if isinstance(val, (dict, list, set)):
            print(f"  {attr}: {type(val).__name__}, len={len(val)}")
            if isinstance(val, list) and len(val) > 0:
                item = val[0]
                if isinstance(item, (dict, set)):
                    print(f"    [0]: {type(item).__name__}, len={len(item)}")
        else:
            print(f"  {attr}: {type(val).__name__}")

# ── Check for cached image features ──────────────────────────────────────
print("\n=== Cached Features ===")
for attr in dir(session):
    if attr.startswith('_'):
        continue
    val = getattr(session, attr, None)
    if isinstance(val, dict):
        for k, v in list(val.items())[:3]:
            if hasattr(v, 'shape') and v.is_cuda:
                mb = v.element_size() * v.nelement() / 1e6
                if mb > 1:
                    print(f"  {attr}[{k}]: {v.shape} {v.dtype} ({mb:.2f} MB)")
            elif isinstance(v, dict):
                for k2, v2 in list(v.items())[:3]:
                    if hasattr(v2, 'shape') and v2.is_cuda:
                        mb = v2.element_size() * v2.nelement() / 1e6
                        if mb > 1:
                            print(f"  {attr}[{k}][{k2}]: {v2.shape} ({mb:.2f} MB)")

del session
gc.collect(); torch.cuda.empty_cache()
print("\nDone. Use the output above to build a targeted prune function for Cell 13.")

✓ Loaded 8 object prompts from /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Initial Prompts/2019_11_05_000002/video1/BoundingBoxes_2019_11_05_000002.txt
  id 3: [196, 299, 436, 420]
  id 4: [462, 438, 678, 526]
  id 6: [927, 402, 1058, 567]
  id 7: [442, 145, 583, 380]
  id 2: [570, 134, 743, 365]
  id 1: [719, 180, 870, 311]
  id 5: [665, 268, 826, 461]
  id 0: [801, 429, 902, 589]
=== VRAM Growth ===
  Frame   1: 3.006 GB
  Frame  10: 3.277 GB
  Frame  20: 3.572 GB
  Frame  30: 3.882 GB
  Frame  40: 4.194 GB
  Frame  50: 4.495 GB
  Frame  60: 4.801 GB
  Growth rate: 30.47 MB/frame
  Projected @ 600 frames: 21.56 GB
  Projected @ 1800 frames (1min @30fps): 58.12 GB

=== Session Attributes ===
  frames_tracked_per_obj: dict, len=8, keys=[0, 1, 2, 3, 4]
  mask_inputs_per_obj: dict, len=8, keys=[0, 1, 2, 3, 4]
  max_vision_features_cache_size = 1
  num_frames = 60
  obj_ids: list, len=8
  obj_with_new_inputs: list, len=0
  output_dict_per_obj: dict, len=8, keys=[0, 1, 2, 3, 4]
  point_i

Cell 13: Benchmark — Student TinyViT + SAM3 Decoder


In [0]:
# CELL 13: BENCHMARK — STUDENT BACKBONE + SAM3 NECK + DECODER (TRUE E2E FPS)
# =========================================================================
# Changes:
#   1. Uses StudentEncoderTinyViT + shared STUDENT_TRANSFORM
#   2. Student runs in bfloat16 via autocast (fair speed comparison)
#   3. DummyBackbone unchanged — still works correctly
# =========================================================================

STUDENT_ANNOT_DIR = osp.join(BASE_DIR, "Mask coordinates", "Student_Backbone_Annotations")
os.makedirs(STUDENT_ANNOT_DIR, exist_ok=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def prune_inference_session(session, keep_last_n=5):
    """
    Remove old non-conditioning frame outputs from session to prevent
    unbounded VRAM growth. SAM3 only needs num_maskmem recent frames
    for memory attention, but the session caches ALL frames by default.
    """
    for obj_idx in range(session.get_obj_num()):
        non_cond = session.output_dict_per_obj[obj_idx].get("non_cond_frame_outputs", {})
        if len(non_cond) <= keep_last_n:
            continue
        sorted_frames = sorted(non_cond.keys())
        frames_to_remove = sorted_frames[:-keep_last_n]
        for frame_key in frames_to_remove:
            del non_cond[frame_key]

class DummyBackbone(nn.Module):
    """
    Replaces SAM3's ViT backbone only.
    Injects student features, lets SAM3's real neck generate FPN.
    """
    def __init__(self):
        super().__init__()
        self.student_features = None  # Set externally before each forward

    def forward(self, *args, **kwargs):
        """Return a fake BaseModelOutput with student features as last_hidden_state."""
        feat = self.student_features  # [B, 1024, 72, 72]
        B, C, H, W = feat.shape
        # vision_encoder.forward reshapes [B, N, C] -> [B, C, H, W] before neck.
        # We return [B, N, C] format so the reshape produces our spatial features.
        fake_lhs = feat.permute(0, 2, 3, 1).reshape(B, H * W, C)  # [B, 5184, 1024]
        from transformers.modeling_outputs import BaseModelOutput
        return BaseModelOutput(last_hidden_state=fake_lhs)

def benchmark_student_backbone(video_path, student_enc, teacher_model, teacher_proc,
                                tracker, device, output_dir, max_frames=None):
    video_id = osp.basename(osp.dirname(video_path))
    frames_dir = prepare_frames_or_path(video_path)
    vid_out = osp.join(output_dir, video_id)
    os.makedirs(vid_out, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"Benchmarking: {video_id} (Student TinyViT + SAM3 Neck/Decoder)")
    print(f"{'='*70}")

    prompts = load_bounding_box_prompts(get_prompt_path_from_video_path(video_path))
    obj_names = sorted(prompts.keys())
    obj_ids = list(range(len(obj_names)))
    input_boxes = prepare_sam3_bbox_format([prompts[n] for n in obj_names])

    frames = sorted([osp.join(frames_dir, f) for f in os.listdir(frames_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if max_frames:
        frames = frames[:max_frames]
    print(f"  Frames: {len(frames)}, Objects: {len(obj_names)}")

    # ── Swap backbone, move original to CPU ───────────────────────────
    dummy_backbone = DummyBackbone().to(device)
    orig_backbone = teacher_model.vision_encoder.backbone
    teacher_model.vision_encoder.backbone = dummy_backbone
    orig_backbone.cpu()
    torch.cuda.empty_cache()

    print(f"  Backbone swapped, original moved to CPU")
    if torch.cuda.is_available():
        print(f"  GPU memory after swap: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    session = teacher_proc.init_video_session(inference_device=device, dtype=DEFAULT_DTYPE)
    annots = {oid: {} for oid in obj_ids}

    tracker.start_video_inference()

    for fi, fp in enumerate(frames):
        frame = Image.open(fp).convert("RGB")
        inputs = teacher_proc(images=frame, device=device, return_tensors="pt")

        if fi == 0:
            teacher_proc.add_inputs_to_inference_session(
                inference_session=session, frame_idx=0,
                obj_ids=obj_ids.copy(), input_boxes=input_boxes,
                original_size=inputs.original_sizes[0])

        # Student forward in bfloat16
        student_input = STUDENT_TRANSFORM(frame).unsqueeze(0).to(device)

        tracker.start_frame()
        with torch.no_grad():
            student_feat = student_enc(student_input)
            del student_input

            dummy_backbone.student_features = student_feat
            del student_feat

            out = teacher_model(inference_session=session, frame=inputs.pixel_values[0])
        tracker.end_frame()

        dummy_backbone.student_features = None

        # ── Post-process masks ────────────────────────────────────────
        masks = teacher_proc.post_process_masks(
            [out.pred_masks], original_sizes=inputs.original_sizes, binarize=False)[0]

        fname = osp.basename(fp)
        for i, oid in enumerate(obj_ids):
            ml = masks[i, -1] if len(masks.shape) == 4 else masks[i]
            mb = (torch.sigmoid(ml) > 0.5).cpu().numpy()
            nz = np.argwhere(mb)
            if len(nz) == 0:
                bbox, area = [0, 0, 0, 0], 0
            else:
                y0, x0 = nz.min(0).tolist()
                y1, x1 = nz.max(0).tolist()
                bbox, area = [x0, y0, x1 - x0, y1 - y0], int(mb.sum())
            tracker.record_mask_quality(mask_area=area)
            annots[oid][fname] = {"bounding_box": bbox, "object_name": obj_names[oid]}

        del inputs, frame, out, masks

        # Prune old session state every 25 frames to prevent unbounded growth
        if (fi + 1) % 25 == 0:
            prune_inference_session(session, keep_last_n=8)
            gc.collect(); torch.cuda.empty_cache()

        if (fi + 1) % 50 == 0:
            elapsed = time.time() - tracker._video_start
            mem_gb = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
            print(f"  [{fi+1}/{len(frames)}] {(fi+1)/elapsed:.1f} FPS  VRAM: {mem_gb:.2f} GB")

    tracker.end_video_inference(len(frames))

    # ── Restore ─────────────────────────────────────────────────────── # Back to fp32 for other cells
    teacher_model.vision_encoder.backbone = orig_backbone.to(device)
    print(f"  Original backbone restored")
    print(f"  {len(frames)} frames @ {tracker._throughput_fps:.2f} FPS")

    for oid, ann in annots.items():
        with open(osp.join(vid_out, f"annotations_id {oid}.json"), "w") as f:
            json.dump(ann, f, indent=2)
    print(f"  Annotations saved to {vid_out}")

    metrics = {}
    try:
        metrics = evaluate_tracking_quality(video_id, vid_out)
        if metrics:
            tracker.add_tracking_quality_metrics(metrics)
            print(f"  MOTA={metrics.get('mota',0)*100:.1f}% "
                  f"IDF1={metrics.get('idf1',0)*100:.1f}%")
    except Exception as e:
        print(f"  Eval error: {e}")

    del session
    gc.collect(); torch.cuda.empty_cache()
    return metrics


# ══════════════════════════════════════════════════════════════════════════
# RUN BENCHMARK ON TEST SET
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "#" * 70)
print("BENCHMARK: STUDENT TinyViT + SAM3 NECK + DECODER (TEST SET)")
print("#" * 70)

if osp.exists(STUDENT_ANNOT_DIR):
    shutil.rmtree(STUDENT_ANNOT_DIR)
os.makedirs(STUDENT_ANNOT_DIR, exist_ok=True)

test_paths = [v for v in BENCHMARK_VIDEOS if osp.basename(osp.dirname(v)) in TEST_VIDEO_IDS]
student_params = student_final.get_num_parameters()
neck_params = sum(p.numel() for p in model.vision_encoder.neck.parameters())
decoder_params = sum(p.numel() for p in model.mask_decoder.parameters())
memory_params = sum(p.numel() for p in model.memory_attention.parameters())
memory_enc_params = sum(p.numel() for p in model.memory_encoder.parameters())

print(f"\nModel breakdown:")
print(f"  Student encoder:    {student_params/1e6:.2f}M")
print(f"  SAM3 Neck (kept):   {neck_params/1e6:.2f}M")
print(f"  SAM3 Decoder (kept):{decoder_params/1e6:.2f}M")
print(f"  Memory Attn (kept): {memory_params/1e6:.2f}M")
print(f"  Memory Enc (kept):  {memory_enc_params/1e6:.2f}M")
total_student_system = (student_params + neck_params + decoder_params +
                        memory_params + memory_enc_params)
print(f"  TOTAL system:       {total_student_system/1e6:.2f}M")
print(f"  vs Teacher:         {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"  Compression:        {sum(p.numel() for p in model.parameters())/total_student_system:.1f}x\n")

all_results = {}

for vp in test_paths:
    vid = osp.basename(osp.dirname(vp))
    tracker = PerformanceTracker("Student_TinyViT_SAM3", log_dir=METRICS_DIR)
    tracker.model_size_metrics = {
        'student_params_M': round(student_params / 1e6, 2),
        'total_system_params_M': round(total_student_system / 1e6, 2),
        'student_size_mb': round(student_params * 4 / 1024**2, 2),
    }

    try:
        metrics = benchmark_student_backbone(
            vp, student_final, model, processor, tracker, device, STUDENT_ANNOT_DIR)
        tracker.print_summary()
        tracker.save_metrics(vid)
        all_results[vid] = {"status": "success", "summary": tracker.get_summary()}
    except Exception as e:
        print(f"Failed {vid}: {e}")
        import traceback; traceback.print_exc()
        all_results[vid] = {"status": "failed", "error": str(e)}

    gc.collect(); torch.cuda.empty_cache()

with open(osp.join(METRICS_DIR, "Student_TinyViT_Test_Results.json"), "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved: Student_TinyViT_Test_Results.json")




######################################################################
BENCHMARK: STUDENT TinyViT + SAM3 NECK + DECODER (TEST SET)
######################################################################

Model breakdown:
  Student encoder:    40.66M
  SAM3 Neck (kept):   7.80M
  SAM3 Decoder (kept):4.22M
  Memory Attn (kept): 5.92M
  Memory Enc (kept):  1.38M
  TOTAL system:       59.98M
  vs Teacher:         465.8M
  Compression:        7.8x


Benchmarking: 2019_12_02_000005 (Student TinyViT + SAM3 Neck/Decoder)
✓ Loaded 8 object prompts from /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Initial Prompts/2019_12_02_000005/video1/BoundingBoxes_2019_12_02_000005.txt
  id 0: [870, 184, 1023, 310]
  id 1: [808, 276, 981, 481]
  id 2: [196, 23, 522, 250]
  id 3: [546, 306, 725, 635]
  id 4: [388, 331, 562, 617]
  id 5: [163, 263, 359, 632]
  id 6: [424, 77, 697, 351]
  id 7: [252, 157, 497, 424]
  Frames: 600, Objects: 8
  Backbone swapped, original moved to CPU
  GPU memory after swap: 1.8

Cell 14: Final Comparison

In [0]:
# CELL 14: TEACHER BASELINE ON TEST SET + COMPARISON + PUBLICATION FIGURES
# =========================================================================
# 1. Run teacher SAM3 on the SAME 4 test videos (no reference numbers)
# 2. Compare side-by-side with student results
# 3. Generate publication-quality figures
# =========================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import numpy as np

FIGURE_DIR = osp.join(BASE_DIR, "Figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════
# PART 1: RUN TEACHER BASELINE ON TEST VIDEOS
# ═══════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════
# PART 1: LOAD EXISTING TEACHER BASELINE (test videos only)
# ═══════════════════════════════════════════════════════════════════════════

teacher_baseline_path = osp.join(METRICS_DIR, "SAM3_baseline_summary.json")
with open(teacher_baseline_path) as f:
    all_teacher = json.load(f)

teacher_results = {}
for vid in TEST_VIDEO_IDS:
    if vid in all_teacher and all_teacher[vid].get("status") == "success":
        teacher_results[vid] = all_teacher[vid]
        tq = all_teacher[vid]["summary"]["tracking_quality_metrics"]
        pm = all_teacher[vid]["summary"]["performance_metrics"]
        print(f"  Loaded teacher: {vid}  MOTA={tq['mota']:.1f}%  FPS={pm['throughput_fps']}")
    else:
        print(f"  WARNING: No teacher result for {vid}")

print(f"  Loaded {len(teacher_results)}/{len(TEST_VIDEO_IDS)} test videos")

# If per-video files not found, try consolidated results file
if not teacher_results:
    for candidate in ["Teacher_Test_Results.json", "SAM3_Baseline_Results.json",
                       "Final_Backbone_Comparison.json"]:
        path = osp.join(METRICS_DIR, candidate)
        if osp.exists(path):
            with open(path) as f:
                data = json.load(f)
            # Handle Final_Backbone_Comparison format
            if "per_video" in data:
                data = data["per_video"]
            for vid in TEST_VIDEO_IDS:
                if vid in data and data[vid].get("status") == "success":
                    teacher_results[vid] = data[vid]
            if teacher_results:
                print(f"  Loaded teacher results from {candidate}")
                break

if not teacher_results:
    print("  WARNING: No teacher results found. Listing available metrics files:")
    for f in sorted(os.listdir(METRICS_DIR)):
        if f.endswith('.json'):
            print(f"    {f}")
    print("  Will use placeholder values for teacher.")
# ═══════════════════════════════════════════════════════════════════════════
# PART 2: LOAD STUDENT RESULTS
# ═══════════════════════════════════════════════════════════════════════════

student_results_path = osp.join(METRICS_DIR, "Student_TinyViT_Test_Results_Improved.json")
with open(student_results_path) as f:
    student_results = json.load(f)


# ═══════════════════════════════════════════════════════════════════════════
# PART 3: BUILD COMPARISON DATA
# ═══════════════════════════════════════════════════════════════════════════

def extract_metrics(results, is_teacher=False):
    """Extract per-video metrics from results dict."""
    rows = []
    for vid in TEST_VIDEO_IDS:
        r = results.get(vid, {})
        if r.get("status") != "success":
            continue
        s = r["summary"]
        pm = s["performance_metrics"]
        tq = s["tracking_quality_metrics"]
        mq = s["mask_quality_metrics"]
        rows.append({
            "video": vid,
            "fps": pm.get("throughput_fps", 0),
            "latency_ms": pm.get("inference_time_ms_mean", 0),
            "gpu_peak_gb": pm.get("gpu_memory_peak_gb", 0),
            "mota": tq.get("mota", 0),
            "idf1": tq.get("idf1", 0),
            "precision": tq.get("precision", 0),
            "recall": tq.get("recall", 0),
            "mask_area": mq.get("mask_areas_mean", 0),
        })
    return rows

teacher_rows = extract_metrics(teacher_results)
student_rows = extract_metrics(student_results)

# Compute averages
def avg(rows, key):
    vals = [r[key] for r in rows if r[key] is not None and r[key] != 0]
    return np.mean(vals) if vals else 0

teacher_total_params = sum(p.numel() for p in model.parameters())
teacher_backbone_params = sum(p.numel() for p in model.vision_encoder.backbone.parameters())
student_total_params = student_final.get_num_parameters()
neck_p = sum(p.numel() for p in model.vision_encoder.neck.parameters())
dec_p = sum(p.numel() for p in model.mask_decoder.parameters())
mem_a_p = sum(p.numel() for p in model.memory_attention.parameters())
mem_e_p = sum(p.numel() for p in model.memory_encoder.parameters())
student_system_params = student_total_params + neck_p + dec_p + mem_a_p + mem_e_p


# ═══════════════════════════════════════════════════════════════════════════
# PART 4: TEXT COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════

print(f"\n{'#'*74}")
print(f"  TEACHER vs STUDENT — FINAL COMPARISON (TEST SET: {len(TEST_VIDEO_IDS)} videos)")
print(f"{'#'*74}")
print(f"  Teacher: SAM3 (ViT-H backbone, 446M)")
print(f"  Student: TinyViT-21M-512 backbone + SAM3 neck/decoder")
print(f"{'─'*74}")
print(f"  {'Metric':<32} {'Teacher (SAM3)':>18} {'Student':>18}")
print(f"{'─'*74}")
print(f"  {'Backbone Params':<32} {teacher_backbone_params/1e6:>15.1f}M  {student_total_params/1e6:>15.2f}M")
print(f"  {'Total System Params':<32} {teacher_total_params/1e6:>15.1f}M  {student_system_params/1e6:>15.2f}M")
print(f"  {'Compression (backbone)':<32} {'1.0x':>18} {teacher_backbone_params/student_total_params:>17.1f}x")
print(f"  {'Compression (total)':<32} {'1.0x':>18} {teacher_total_params/student_system_params:>17.1f}x")
print(f"{'─'*74}")
print(f"  {'Throughput (FPS)':<32} {avg(teacher_rows,'fps'):>15.2f}FPS {avg(student_rows,'fps'):>15.2f}FPS")
print(f"  {'Latency (ms/frame)':<32} {avg(teacher_rows,'latency_ms'):>15.1f}ms  {avg(student_rows,'latency_ms'):>15.1f}ms")
print(f"  {'GPU Peak Memory':<32} {avg(teacher_rows,'gpu_peak_gb'):>14.3f}GB  {avg(student_rows,'gpu_peak_gb'):>14.3f}GB")
print(f"{'─'*74}")
print(f"  {'MOTA':<32} {avg(teacher_rows,'mota'):>17.1f}%  {avg(student_rows,'mota'):>17.1f}%")
print(f"  {'IDF1':<32} {avg(teacher_rows,'idf1'):>17.1f}%  {avg(student_rows,'idf1'):>17.1f}%")
print(f"  {'Precision':<32} {avg(teacher_rows,'precision'):>17.1f}%  {avg(student_rows,'precision'):>17.1f}%")
print(f"  {'Recall':<32} {avg(teacher_rows,'recall'):>17.1f}%  {avg(student_rows,'recall'):>17.1f}%")
print(f"{'─'*74}")

# Delta row
mota_delta = avg(student_rows, 'mota') - avg(teacher_rows, 'mota')
idf1_delta = avg(student_rows, 'idf1') - avg(teacher_rows, 'idf1')
fps_ratio = avg(student_rows, 'fps') / avg(teacher_rows, 'fps') if avg(teacher_rows, 'fps') > 0 else 0
print(f"  {'MOTA Delta':<32} {'—':>18} {mota_delta:>+17.1f}%")
print(f"  {'IDF1 Delta':<32} {'—':>18} {idf1_delta:>+17.1f}%")
print(f"  {'Speed Ratio':<32} {'1.0x':>18} {fps_ratio:>17.2f}x")
print(f"{'#'*74}")

# Per-video table
print(f"\n  Per-Video Breakdown:")
print(f"  {'Video':<22} {'':^3} {'FPS':>6} {'MOTA':>7} {'IDF1':>7} {'Prec':>7} {'Rec':>7} {'GPU GB':>7}")
print(f"  {'─'*70}")
for vid in TEST_VIDEO_IDS:
    tr = next((r for r in teacher_rows if r['video'] == vid), None)
    sr = next((r for r in student_rows if r['video'] == vid), None)
    if tr:
        print(f"  {vid:<22} {'T':^3} {tr['fps']:>6.2f} {tr['mota']:>7.1f} {tr['idf1']:>7.1f} "
              f"{tr['precision']:>7.1f} {tr['recall']:>7.1f} {tr['gpu_peak_gb']:>7.3f}")
    if sr:
        print(f"  {'':22} {'S':^3} {sr['fps']:>6.2f} {sr['mota']:>7.1f} {sr['idf1']:>7.1f} "
              f"{sr['precision']:>7.1f} {sr['recall']:>7.1f} {sr['gpu_peak_gb']:>7.3f}")
print(f"  {'─'*70}")
print(f"  T = Teacher (SAM3), S = Student (TinyViT-21M)")


# ═══════════════════════════════════════════════════════════════════════════
# PART 5: PUBLICATION FIGURES
# ═══════════════════════════════════════════════════════════════════════════

# ── Color scheme (professional, colorblind-friendly) ──
C_TEACHER = '#2563EB'   # blue
C_STUDENT = '#F59E0B'   # amber
C_BG      = '#FAFBFC'
C_GRID    = '#E5E7EB'
C_TEXT    = '#1F2937'

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.facecolor': C_BG,
    'figure.facecolor': 'white',
    'axes.edgecolor': C_GRID,
    'axes.grid': True,
    'grid.color': C_GRID,
    'grid.linewidth': 0.5,
    'text.color': C_TEXT,
    'axes.labelcolor': C_TEXT,
    'xtick.color': C_TEXT,
    'ytick.color': C_TEXT,
})

short_ids = [v.replace('2019_', '').replace('_000', '_') for v in TEST_VIDEO_IDS]


# ── Figure 1: Tracking Quality Comparison (grouped bar) ──────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={'width_ratios': [3, 2]})
fig.suptitle('Backbone Distillation: Teacher vs Student Tracking Quality',
             fontsize=15, fontweight='bold', y=1.02)

# Panel A: Per-video MOTA & IDF1
ax = axes[0]
x = np.arange(len(TEST_VIDEO_IDS))
w = 0.18

t_mota = [next((r['mota'] for r in teacher_rows if r['video'] == v), 0) for v in TEST_VIDEO_IDS]
s_mota = [next((r['mota'] for r in student_rows if r['video'] == v), 0) for v in TEST_VIDEO_IDS]
t_idf1 = [next((r['idf1'] for r in teacher_rows if r['video'] == v), 0) for v in TEST_VIDEO_IDS]
s_idf1 = [next((r['idf1'] for r in student_rows if r['video'] == v), 0) for v in TEST_VIDEO_IDS]

bars1 = ax.bar(x - 1.5*w, t_mota, w, label='Teacher MOTA', color=C_TEACHER, alpha=0.85, edgecolor='white')
bars2 = ax.bar(x - 0.5*w, s_mota, w, label='Student MOTA', color=C_STUDENT, alpha=0.85, edgecolor='white')
bars3 = ax.bar(x + 0.5*w, t_idf1, w, label='Teacher IDF1', color=C_TEACHER, alpha=0.45, edgecolor='white',
               hatch='///')
bars4 = ax.bar(x + 1.5*w, s_idf1, w, label='Student IDF1', color=C_STUDENT, alpha=0.45, edgecolor='white',
               hatch='///')

ax.set_ylabel('Score (%)')
ax.set_xlabel('Test Video')
ax.set_xticks(x)
ax.set_xticklabels(short_ids, fontsize=9, rotation=15, ha='right')
ax.set_ylim(70, 102)
ax.legend(fontsize=8.5, ncol=2, loc='lower left', framealpha=0.9)
ax.set_title('(a) Per-Video Tracking Metrics', fontsize=12)

# Add value labels on bars
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.annotate(f'{h:.1f}', xy=(bar.get_x() + bar.get_width()/2, h),
                       xytext=(0, 3), textcoords='offset points',
                       ha='center', va='bottom', fontsize=7)

# Panel B: Average metrics radar-style (horizontal bar comparison)
ax2 = axes[1]
metrics_names = ['MOTA', 'IDF1', 'Precision', 'Recall']
t_vals = [avg(teacher_rows, k) for k in ['mota', 'idf1', 'precision', 'recall']]
s_vals = [avg(student_rows, k) for k in ['mota', 'idf1', 'precision', 'recall']]

y_pos = np.arange(len(metrics_names))
bar_h = 0.3

ax2.barh(y_pos + bar_h/2, t_vals, bar_h, label='Teacher (SAM3)',
         color=C_TEACHER, alpha=0.85, edgecolor='white')
ax2.barh(y_pos - bar_h/2, s_vals, bar_h, label='Student (TinyViT)',
         color=C_STUDENT, alpha=0.85, edgecolor='white')

for i, (tv, sv) in enumerate(zip(t_vals, s_vals)):
    ax2.text(tv + 0.5, i + bar_h/2, f'{tv:.1f}%', va='center', fontsize=9, color=C_TEACHER, fontweight='bold')
    ax2.text(sv + 0.5, i - bar_h/2, f'{sv:.1f}%', va='center', fontsize=9, color=C_STUDENT, fontweight='bold')

ax2.set_yticks(y_pos)
ax2.set_yticklabels(metrics_names)
ax2.set_xlim(70, 105)
ax2.set_xlabel('Score (%)')
ax2.legend(fontsize=9, loc='lower right', framealpha=0.9)
ax2.set_title('(b) Average Tracking Quality', fontsize=12)

plt.tight_layout()
fig.savefig(osp.join(FIGURE_DIR, 'fig1_tracking_quality.png'), dpi=300, bbox_inches='tight')
fig.savefig(osp.join(FIGURE_DIR, 'fig1_tracking_quality.pdf'), bbox_inches='tight')
plt.close()
print(f"  Saved fig1_tracking_quality.png/pdf")


# ── Figure 2: Model Efficiency Comparison ─────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Efficiency: SAM3 Teacher vs Distilled Student',
             fontsize=15, fontweight='bold', y=1.02)

# Panel A: Parameter count breakdown
ax = axes[0]
teacher_parts = {
    'Backbone\n(ViT-H)': teacher_backbone_params/1e6,
    'Neck': neck_p/1e6,
    'Decoder': dec_p/1e6,
    'Memory': (mem_a_p + mem_e_p)/1e6,
}
student_parts = {
    'Backbone\n(TinyViT)': student_total_params/1e6,
    'Neck': neck_p/1e6,
    'Decoder': dec_p/1e6,
    'Memory': (mem_a_p + mem_e_p)/1e6,
}

labels = list(teacher_parts.keys())
t_sizes = list(teacher_parts.values())
s_sizes = list(student_parts.values())
x = np.arange(len(labels))
w = 0.35

b1 = ax.bar(x - w/2, t_sizes, w, label='Teacher', color=C_TEACHER, alpha=0.85, edgecolor='white')
b2 = ax.bar(x + w/2, s_sizes, w, label='Student', color=C_STUDENT, alpha=0.85, edgecolor='white')

for bar in b1:
    h = bar.get_height()
    if h > 5:
        ax.text(bar.get_x() + bar.get_width()/2, h + 3, f'{h:.1f}M',
                ha='center', fontsize=8, fontweight='bold')
for bar in b2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 3, f'{h:.1f}M',
            ha='center', fontsize=8, fontweight='bold')

ax.set_ylabel('Parameters (M)')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.legend(fontsize=9)
ax.set_title('(a) Parameter Count by Component', fontsize=12)
ax.set_yscale('log')
ax.set_ylim(1, 800)

# Panel B: Latency comparison
ax2 = axes[1]
models = ['Teacher\n(SAM3)', 'Student\n(TinyViT)']
fps_vals = [avg(teacher_rows, 'fps'), avg(student_rows, 'fps')]
colors = [C_TEACHER, C_STUDENT]

bars = ax2.bar(models, fps_vals, width=0.5, color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, fps_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.1,
             f'{val:.2f} FPS\n({1000/val:.0f} ms/frame)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Throughput (FPS)')
ax2.set_title('(b) Inference Speed', fontsize=12)
ax2.set_ylim(0, max(fps_vals) * 1.4)

# Panel C: GPU Memory
ax3 = axes[2]
gpu_vals = [avg(teacher_rows, 'gpu_peak_gb'), avg(student_rows, 'gpu_peak_gb')]

bars = ax3.bar(models, gpu_vals, width=0.5, color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, gpu_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.1,
             f'{val:.2f} GB',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add edge device reference lines
ax3.axhline(y=8, color='#EF4444', linestyle='--', alpha=0.6, linewidth=1)
ax3.text(1.35, 8.1, 'Jetson Orin NX 8GB', fontsize=7, color='#EF4444', ha='right')
ax3.axhline(y=16, color='#10B981', linestyle='--', alpha=0.6, linewidth=1)
ax3.text(1.35, 16.1, 'Jetson Orin NX 16GB', fontsize=7, color='#10B981', ha='right')

ax3.set_ylabel('Peak GPU Memory (GB)')
ax3.set_title('(c) GPU Memory Usage', fontsize=12)
ax3.set_ylim(0, max(gpu_vals) * 1.15 + 2)

plt.tight_layout()
fig.savefig(osp.join(FIGURE_DIR, 'fig2_model_efficiency.png'), dpi=300, bbox_inches='tight')
fig.savefig(osp.join(FIGURE_DIR, 'fig2_model_efficiency.pdf'), bbox_inches='tight')
plt.close()
print(f"  Saved fig2_model_efficiency.png/pdf")


# ── Figure 3: Quality vs Efficiency Scatter ───────────────────────────────

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('Tracking Quality vs Model Size',
             fontsize=15, fontweight='bold', y=1.01)

# Teacher point
ax.scatter(teacher_total_params/1e6, avg(teacher_rows, 'mota'),
           s=300, c=C_TEACHER, marker='D', edgecolors='white', linewidth=2,
           zorder=5, label='Teacher (SAM3 ViT-H)')

# Student point
ax.scatter(student_system_params/1e6, avg(student_rows, 'mota'),
           s=300, c=C_STUDENT, marker='o', edgecolors='white', linewidth=2,
           zorder=5, label='Student (TinyViT-21M)')

# Annotations
ax.annotate(f"SAM3\n{teacher_total_params/1e6:.0f}M params\n"
            f"MOTA: {avg(teacher_rows,'mota'):.1f}%\n"
            f"{avg(teacher_rows,'fps'):.1f} FPS",
            xy=(teacher_total_params/1e6, avg(teacher_rows, 'mota')),
            xytext=(-120, -50), textcoords='offset points',
            fontsize=9, ha='center',
            arrowprops=dict(arrowstyle='->', color=C_TEACHER, lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor=C_TEACHER, alpha=0.1))

ax.annotate(f"Student\n{student_system_params/1e6:.0f}M params\n"
            f"MOTA: {avg(student_rows,'mota'):.1f}%\n"
            f"{avg(student_rows,'fps'):.2f} FPS",
            xy=(student_system_params/1e6, avg(student_rows, 'mota')),
            xytext=(100, 40), textcoords='offset points',
            fontsize=9, ha='center',
            arrowprops=dict(arrowstyle='->', color=C_STUDENT, lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor=C_STUDENT, alpha=0.1))

# Compression arrow
ax.annotate('', xy=(student_system_params/1e6, avg(student_rows, 'mota')),
            xytext=(teacher_total_params/1e6, avg(teacher_rows, 'mota')),
            arrowprops=dict(arrowstyle='->', color='#6B7280', lw=1.5, linestyle='--'))
mid_x = (teacher_total_params/1e6 + student_system_params/1e6) / 2
mid_y = (avg(teacher_rows, 'mota') + avg(student_rows, 'mota')) / 2
ax.text(mid_x, mid_y + 1.5,
        f'{teacher_total_params/student_system_params:.1f}x compression',
        fontsize=10, ha='center', color='#6B7280', fontstyle='italic')

ax.set_xlabel('Total System Parameters (M)', fontsize=12)
ax.set_ylabel('MOTA (%)', fontsize=12)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, teacher_total_params/1e6 * 1.15)

plt.tight_layout()
fig.savefig(osp.join(FIGURE_DIR, 'fig3_quality_vs_size.png'), dpi=300, bbox_inches='tight')
fig.savefig(osp.join(FIGURE_DIR, 'fig3_quality_vs_size.pdf'), bbox_inches='tight')
plt.close()
print(f"  Saved fig3_quality_vs_size.png/pdf")


# ── Figure 4: VRAM Growth Over Time ──────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('GPU Memory Growth During 600-Frame Inference',
             fontsize=14, fontweight='bold', y=1.01)

# Student VRAM data (from the benchmark output)
student_frames = np.arange(50, 650, 50)
# Approximate from the logs: starts ~2.4 GB, linear growth ~5.8 MB/frame
student_vram = np.array([2.40, 2.69, 2.98, 3.26, 3.55, 3.84,
                         4.13, 4.41, 4.70, 4.98, 5.26, 5.55])

ax.plot(student_frames, student_vram, 'o-', color=C_STUDENT, linewidth=2,
        markersize=6, label='Student (TinyViT-21M)', zorder=5)

# Fit linear trend
slope, intercept = np.polyfit(student_frames, student_vram, 1)
ax.plot([0, 600], [intercept, intercept + slope * 600], '--',
        color=C_STUDENT, alpha=0.4, linewidth=1)
ax.text(620, student_vram[-1], f'+{slope*1000:.1f} MB/frame', fontsize=9,
        color=C_STUDENT, va='center')

# Edge device limits
ax.axhline(y=8, color='#EF4444', linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(605, 8.15, 'Orin NX 8GB', fontsize=8, color='#EF4444')
ax.axhline(y=16, color='#10B981', linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(605, 16.15, 'Orin NX 16GB', fontsize=8, color='#10B981')

# Project where 8GB limit would be hit
if slope > 0:
    frame_at_8gb = (8 - intercept) / slope
    ax.axvline(x=frame_at_8gb, color='#EF4444', linestyle=':', alpha=0.4)
    ax.text(frame_at_8gb, 1.5, f'OOM @ frame {frame_at_8gb:.0f}',
            fontsize=8, color='#EF4444', ha='center', rotation=90)

ax.set_xlabel('Frame Number')
ax.set_ylabel('GPU Memory (GB)')
ax.set_xlim(0, 700)
ax.set_ylim(0, 18)
ax.legend(fontsize=10)

plt.tight_layout()
fig.savefig(osp.join(FIGURE_DIR, 'fig4_vram_growth.png'), dpi=300, bbox_inches='tight')
fig.savefig(osp.join(FIGURE_DIR, 'fig4_vram_growth.pdf'), bbox_inches='tight')
plt.close()
print(f"  Saved fig4_vram_growth.png/pdf")


# ═══════════════════════════════════════════════════════════════════════════
# PART 6: SAVE FINAL COMPARISON JSON
# ═══════════════════════════════════════════════════════════════════════════

comparison = {
    "test_videos": TEST_VIDEO_IDS,
    "teacher": {
        "backbone": "ViT-H (SAM3)",
        "backbone_params_M": teacher_backbone_params / 1e6,
        "total_params_M": teacher_total_params / 1e6,
        "avg_fps": avg(teacher_rows, 'fps'),
        "avg_latency_ms": avg(teacher_rows, 'latency_ms'),
        "avg_gpu_peak_gb": avg(teacher_rows, 'gpu_peak_gb'),
        "avg_mota": avg(teacher_rows, 'mota'),
        "avg_idf1": avg(teacher_rows, 'idf1'),
        "avg_precision": avg(teacher_rows, 'precision'),
        "avg_recall": avg(teacher_rows, 'recall'),
    },
    "student": {
        "backbone": "TinyViT-21M-512 + FPN projection",
        "backbone_params_M": student_total_params / 1e6,
        "total_system_params_M": student_system_params / 1e6,
        "backbone_compression": teacher_backbone_params / student_total_params,
        "total_compression": teacher_total_params / student_system_params,
        "avg_fps": avg(student_rows, 'fps'),
        "avg_latency_ms": avg(student_rows, 'latency_ms'),
        "avg_gpu_peak_gb": avg(student_rows, 'gpu_peak_gb'),
        "avg_mota": avg(student_rows, 'mota'),
        "avg_idf1": avg(student_rows, 'idf1'),
        "avg_precision": avg(student_rows, 'precision'),
        "avg_recall": avg(student_rows, 'recall'),
        "mota_delta": avg(student_rows, 'mota') - avg(teacher_rows, 'mota'),
        "idf1_delta": avg(student_rows, 'idf1') - avg(teacher_rows, 'idf1'),
    },
    "per_video_teacher": {r['video']: r for r in teacher_rows},
    "per_video_student": {r['video']: r for r in student_rows},
}

comp_path = osp.join(METRICS_DIR, "Final_TinyViT_Comparison.json")
with open(comp_path, "w") as f:
    json.dump(comparison, f, indent=2, default=str)
print(f"\n  Saved: {comp_path}")

print(f"\n{'='*74}")
print(f"  FIGURES saved to: {FIGURE_DIR}")
print(f"  - fig1_tracking_quality.png/pdf")
print(f"  - fig2_model_efficiency.png/pdf")
print(f"  - fig3_quality_vs_size.png/pdf")
print(f"  - fig4_vram_growth.png/pdf")
print(f"{'='*74}")
print(f"\n  NEXT STEPS:")
print(f"  1. Fix VRAM leak: prune session memory (currently grows +5.8 MB/frame)")
print(f"  2. Distill neck + decoder for true edge deployment")
print(f"  3. QAT after ALL distillation stages complete")
print(f"  4. Export to TensorRT/ONNX for Jetson deployment")
print(f"{'='*74}")

  Loaded teacher: 2019_12_02_000005  MOTA=91.0%  FPS=1.0716
  Loaded teacher: 2019_12_02_000208  MOTA=98.4%  FPS=1.0747
  Loaded teacher: 2019_12_10_000060  MOTA=100.0%  FPS=1.0778
  Loaded teacher: 2019_12_10_000078  MOTA=86.5%  FPS=1.0893
  Loaded 4/4 test videos

##########################################################################
  TEACHER vs STUDENT — FINAL COMPARISON (TEST SET: 4 videos)
##########################################################################
  Teacher: SAM3 (ViT-H backbone, 446M)
  Student: TinyViT-21M-512 backbone + SAM3 neck/decoder
──────────────────────────────────────────────────────────────────────────
  Metric                               Teacher (SAM3)            Student
──────────────────────────────────────────────────────────────────────────
  Backbone Params                            446.2M            40.66M
  Total System Params                        465.8M            59.98M
  Compression (backbone)                         1.0x           